# 🌳 Decision Tree Model Training

This notebook trains the Decision Tree baseline model.

**Prerequisites**: Run `01_preprocessing.ipynb` first to generate preprocessed data.

In [1]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import pickle

# Ensure the project is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

# Project modules
from src.config import *
from src.utils import get_logger

logger = get_logger('Decision Tree')
print('✅ Imports successful')

✅ Imports successful


## 1. Load Preprocessed Data

In [2]:
# Load preprocessed data from file
preprocess_file = os.path.join(MODEL_DIR, 'preprocessed_data.pkl')

if not os.path.exists(preprocess_file):
    raise FileNotFoundError(f'Please run 01_preprocessing.ipynb first to generate {preprocess_file}')

with open(preprocess_file, 'rb') as f:
    prep = pickle.load(f)

X_train = prep['X_train']
X_val = prep['X_val']
X_test = prep['X_test']
y_train = prep['y_train']
y_val = prep['y_val']
y_test = prep['y_test']

print(f'✅ Preprocessed data loaded')
print(f'   Train: {X_train.shape}')
print(f'   Val:   {X_val.shape}')
print(f'   Test:  {X_test.shape}')

✅ Preprocessed data loaded
   Train: (124012, 217)
   Val:   (26575, 217)
   Test:  (26575, 217)


## 2. Train Decision Tree

In [3]:
import sys
import os

# Get the absolute path to the project root (one level up from 'notebook')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to the Python path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added: {project_root}")

Project root added: c:\Users\Lekshmi Priya\OneDrive\Documents\GitHub\Credit-Card-Fraud-Detection-System


In [4]:
from src.training import train_decision_tree

dt_model = train_decision_tree(X_train, y_train, X_val, y_val, use_smote=True)
print('\n✅ Decision Tree training complete')

20:49:18 | Training             | INFO    | ==================================================
20:49:18 | Training             | INFO    |   🌳 Training Decision Tree
20:49:18 | Training             | INFO    | ==================================================
20:49:27 | Training             | INFO    |   SMOTE Resampling:
20:49:27 | Training             | INFO    |     Before: 119,573 legit, 4,439 fraud (1:26.9)
20:49:27 | Training             | INFO    |     After:  119,573 legit, 59,786 fraud (1:2.0)
20:49:27 | Training             | INFO    |     Synthetic samples: 55,347
20:49:27 | Training             | INFO    | ⏳ Starting: Decision Tree training
20:49:27 | Models               | INFO    |   🌳 Decision Tree created (max_depth=12, class_weight=balanced)
20:49:42 | Training             | INFO    | ✅ Finished: Decision Tree training (15.0s)
20:49:42 | Training             | INFO    |   Val Accuracy: 0.9369
20:49:42 | Training             | INFO    |   Val F1-Score: 0.3762
20:49:42 


✅ Decision Tree training complete


## 3. Evaluate on Test Set

## 3.5 Feature Importance Visualization

In [5]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Feature importance
feature_importance = dt_model.feature_importances_
feature_names = prep['feature_names'] if 'feature_names' in prep else [f'Feature {i}' for i in range(len(feature_importance))]

# Sort by importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance'], color='#2ecc71', edgecolor='black')
ax.set_xlabel('Importance Score', fontweight='bold')
ax.set_title('Decision Tree - Top 20 Most Important Features', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '02_dt_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance plot saved')

✅ Feature importance plot saved


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Predictions
y_pred = dt_model.predict(X_test)
y_pred_proba = dt_model.predict_proba(X_test)[:, 1]

# Metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
}

print('\n' + '='*50)
print('Decision Tree - Test Set Performance')
print('='*50)
for metric, value in metrics.items():
    print(f'{metric:15s}: {value:.4f}')
print('='*50)


Decision Tree - Test Set Performance
Accuracy       : 0.9366
Precision      : 0.2929
Recall         : 0.5457
F1-Score       : 0.3812
ROC-AUC        : 0.8390


## 4. Save Model and Results

In [7]:
# Save results (predictions only, not full model)
dt_results = {
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba,
    'metrics': metrics,
}

results_file = os.path.join(MODEL_DIR, 'decision_tree_results.pkl')
with open(results_file, 'wb') as f:
    pickle.dump(dt_results, f)

print(f'\n✅ Results saved to {results_file}')
print('\n📝 Next: Run 03_train_xgboost.ipynb or 04_train_hgnn.ipynb')



✅ Results saved to c:\Users\Lekshmi Priya\OneDrive\Documents\GitHub\Credit-Card-Fraud-Detection-System\models\decision_tree_results.pkl

📝 Next: Run 03_train_xgboost.ipynb or 04_train_hgnn.ipynb


In [8]:
from sklearn.metrics import confusion_matrix

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

# Total fraud cases in test set
total_fraud_cases = tp + fn

# Fraud cases correctly detected
fraud_correctly_detected = tp

# Fraud detection rate
fraud_detection_rate = (fraud_correctly_detected / total_fraud_cases) * 100 if total_fraud_cases > 0 else 0

print('\n' + '='*70)
print('FRAUD DETECTION ANALYSIS - Decision Tree MODEL')
print('='*70)
print(f'\n📊 CONFUSION MATRIX BREAKDOWN:')
print(f'   True Negatives (TN):  {tn:6d}  (Correctly identified non-frauds)')
print(f'   False Positives (FP): {fp:6d}  (Non-frauds incorrectly flagged as fraud)')
print(f'   False Negatives (FN): {fn:6d}  (Frauds missed by the model)')
print(f'   True Positives (TP):  {tp:6d}  (Correctly identified frauds)')

print(f'\n🎯 FRAUD DETECTION RESULTS:')
print(f'   Total fraud cases in test set:  {total_fraud_cases}')
print(f'   Fraud cases detected correctly: {fraud_correctly_detected}')
print(f'   Fraud cases MISSED:             {fn}')
print(f'   Fraud detection rate:           {fraud_detection_rate:.2f}%')

print(f'\n⚠️  FALSE POSITIVES (Type I Errors):')
print(f'   Non-fraudulent transactions flagged as fraud: {fp}')
print(f'   False positive rate: {(fp / (tn + fp)) * 100:.2f}%' if (tn + fp) > 0 else '   False positive rate: N/A')

print('\n' + '='*70)


FRAUD DETECTION ANALYSIS - Decision Tree MODEL

📊 CONFUSION MATRIX BREAKDOWN:
   True Negatives (TN):   24371  (Correctly identified non-frauds)
   False Positives (FP):   1253  (Non-frauds incorrectly flagged as fraud)
   False Negatives (FN):    432  (Frauds missed by the model)
   True Positives (TP):     519  (Correctly identified frauds)

🎯 FRAUD DETECTION RESULTS:
   Total fraud cases in test set:  951
   Fraud cases detected correctly: 519
   Fraud cases MISSED:             432
   Fraud detection rate:           54.57%

⚠️  FALSE POSITIVES (Type I Errors):
   Non-fraudulent transactions flagged as fraud: 1253
   False positive rate: 4.89%

